# Project Risk Prediction - Final Training Workflow

This top section is the authoritative workflow used by the application. It matches `train_risk_model.py` and the generated `ML_REPORT.md`:

1. Analyse exploratoire (EDA)
2. Preprocessing
3. Baseline modelling with 5 models: KNN, Logistic Regression, Decision Tree, Random Forest, XGBoost
4. Decision Tree visualization
5. Random Forest feature importance
6. SelectKBest feature selection
7. GridSearchCV fine-tuning
8. Final model selection on a held-out test split
9. Save model artifacts and write the report

The model predicts project risk level (`low`, `medium`, `high`) from selected direct project-risk drivers, not from all CSV columns.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ML_DIR = Path.cwd()
if not (ML_DIR / "project_risk_raw_dataset.csv").exists():
    ML_DIR = Path(r"D:/Hamza pfe/pfe/ml")

CSV_PATH = ML_DIR / "project_risk_raw_dataset.csv"
REPORT_PATH = ML_DIR / "ML_REPORT.md"
MODEL_META_PATH = ML_DIR / "models" / "risk_meta.json"

print(f"ML directory: {ML_DIR}")
print(f"Dataset: {CSV_PATH}")
print(f"Report: {REPORT_PATH}")

## EDA and Selected Features

The raw CSV has 4,000 rows and 51 columns. We intentionally select a compact set of direct project-risk drivers instead of training on every column. The production script then adds three engineered features:

- `budget_per_person = budget_usd / team_size`
- `timeline_pressure = complexity_score / duration_months`
- `team_effectiveness = team_experience * success_rate`

In [ ]:
df_raw = pd.read_csv(CSV_PATH)
print(f"Rows: {len(df_raw)}")
print(f"Columns: {len(df_raw.columns)}")
print("\nTarget distribution:")
print(df_raw["Risk_Level"].value_counts())

selected_csv_columns = [
    "Team_Size", "Project_Budget_USD", "Estimated_Timeline_Months",
    "Complexity_Score", "Stakeholder_Count", "Past_Similar_Projects",
    "External_Dependencies_Count", "Change_Request_Frequency",
    "Team_Turnover_Rate", "Vendor_Reliability_Score", "Schedule_Pressure",
    "Budget_Utilization_Rate", "Resource_Availability",
    "Previous_Delivery_Success_Rate", "Technical_Debt_Level",
    "Team_Experience_Level", "Requirement_Stability",
    "Risk_Management_Maturity", "Documentation_Quality",
]

print("\nSelected input columns:")
for col in selected_csv_columns:
    print("-", col)

df_raw[selected_csv_columns + ["Risk_Level"]].head()

## Execute the Production Training Pipeline

The notebook executes the same script used by the backend model artifacts. This script performs:

- EDA summary
- Preprocessing and encoding
- 5 baseline models: KNN, Logistic Regression, Decision Tree, Random Forest, XGBoost
- SelectKBest feature selection
- GridSearchCV fine-tuning
- Decision Tree visualization and Random Forest feature importance
- Final model save and `ML_REPORT.md` regeneration

In [ ]:
import subprocess
import sys

command = [sys.executable, str(ML_DIR / "train_risk_model.py")]
print("Running:", " ".join(command))

result = subprocess.run(
    command,
    cwd=str(ML_DIR),
    text=True,
    capture_output=True,
)

print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"Training failed with exit code {result.returncode}")

## Results and Artifacts

After training, inspect the saved metadata and generated files. The metadata records the winning algorithm, held-out test accuracy, all candidate features, and the SelectKBest-selected feature subset.

In [ ]:
with open(MODEL_META_PATH, "r", encoding="utf-8") as handle:
    meta = json.load(handle)

print("Algorithm:", meta.get("algorithm"))
print("Test accuracy:", f"{meta.get('accuracy', 0) * 100:.2f}%")
print("Rows used:", meta.get("training_rows"))
print("Features:", len(meta.get("features", [])))
print("Selected features:", meta.get("selected_features"))

outputs_dir = ML_DIR / "outputs"
print("\nGenerated artifacts:")
for path in sorted(outputs_dir.glob("*")):
    print("-", path.name)

## Visualizations

The saved plots document the requested interpretation steps: SelectKBest scores, Decision Tree visualization, Random Forest feature importance, and the final confusion matrix.

In [ ]:
from IPython.display import Image, display

for filename in [
    "selectkbest_scores.png",
    "decision_tree_visualization.png",
    "random_forest_feature_importance.png",
    "final_confusion_matrix.png",
]:
    path = ML_DIR / "outputs" / filename
    if path.exists():
        print(filename)
        display(Image(filename=str(path)))

## Conclusion

The final application model is selected by held-out test accuracy after comparing the five requested baselines and tuned versions. The training script currently selects XGBoost, saves the complete preprocessing + SelectKBest + model pipeline, and writes the frontend/backend metadata used at inference time.

The practical result is a real-data model trained on 3,209 clean CSV rows with 19 direct project-risk inputs plus 3 engineered features. The final held-out test accuracy is 57.48%.